In [5]:
import os
import numpy as np
import pyvista as pv
import json
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

In [6]:
BASE_LATTICE_DIR = Path(r"C:\Users\jkowa\OneDrive\Pulpit\burns\LatticeData")
BASE_OUT_DIR = Path(r"C:\Users\jkowa\OneDrive\Pulpit\burns\preprocessed")

CYTOKINE_NAMES = ["il8", "il1", "il6", "il10", "tnf", "tgf"]
N_FEATURES = 6
N_TIMESTEPS = 101

In [7]:
def process_mesh_resolution(folder_name):
    data_path = os.path.join(BASE_LATTICE_DIR, folder_name)
    
    folder_clean = folder_name.replace("LatticeData", "").replace("(", "").replace(")", "").strip()
    out_path = os.path.join(BASE_OUT_DIR, folder_clean)
    os.makedirs(out_path, exist_ok=True)

    vtk_files = sorted([f for f in os.listdir(data_path) if f.lower().endswith(".vtk")])
    if not vtk_files:
        print(f"Skipping {folder_name}: No .vtk files found.")
        return

    sample_mesh = pv.read(os.path.join(data_path, vtk_files[0]))
    grid_size = int(np.sqrt(sample_mesh.n_points))
    
    print(f"Processing {folder_name} (Grid: {grid_size}x{grid_size})")

    cytokine_fields = np.zeros((N_TIMESTEPS, grid_size, grid_size, N_FEATURES))
    coords = None

    # normalization
    for i, file in enumerate(vtk_files[:N_TIMESTEPS]):
        mesh = pv.read(os.path.join(data_path, file))
        for j, ck in enumerate(CYTOKINE_NAMES):
            cytokine_fields[i, :, :, j] = mesh.point_data[ck].reshape(grid_size, grid_size)
        
        if coords is None:
            # normalizing spatial coordinates to [0, 1] range
            raw_coords = np.array(mesh.points).reshape(grid_size, grid_size, 3)[:, :, :2]
            coords = (raw_coords - raw_coords.min()) / (raw_coords.max() - raw_coords.min())

    # global scaling
    scaler = MinMaxScaler()
    flat_fields = cytokine_fields.reshape(-1, N_FEATURES)
    scaled_fields = scaler.fit_transform(flat_fields).reshape(N_TIMESTEPS, grid_size, grid_size, N_FEATURES)
    
    # generating model tensors 
    window = 2 # like before
    n_samples = N_TIMESTEPS - window
    t_norm = np.linspace(0, 1, N_TIMESTEPS)

    X_lstm = np.array([scaled_fields[i-window:i] for i in range(window, N_TIMESTEPS)])
    Y_target = np.array([scaled_fields[i] for i in range(window, N_TIMESTEPS)])

    X_branch = X_lstm.reshape(n_samples, grid_size, grid_size, -1)
    X_trunk = np.zeros((n_samples, grid_size * grid_size, 3))
    for s in range(n_samples):
        X_trunk[s, :, :2] = coords.reshape(-1, 2)
        X_trunk[s, :, 2] = t_norm[s + window]

    # PINN pointwise tensors
    pinn_in, pinn_out = [], []
    for t in range(N_TIMESTEPS):
        xy = coords.reshape(-1, 2)
        t_col = np.full((grid_size * grid_size, 1), t_norm[t])
        pinn_in.append(np.hstack([xy, t_col]))
        pinn_out.append(scaled_fields[t].reshape(-1, N_FEATURES))
    
    X_pinn, Y_pinn = np.vstack(pinn_in), np.vstack(pinn_out)

    np.save(os.path.join(out_path, "X_lstm.npy"), X_lstm)
    np.save(os.path.join(out_path, "Y_target.npy"), Y_target)
    np.save(os.path.join(out_path, "X_branch.npy"), X_branch)
    np.save(os.path.join(out_path, "X_trunk.npy"), X_trunk)
    np.save(os.path.join(out_path, "X_pinn.npy"), X_pinn)
    np.save(os.path.join(out_path, "Y_pinn.npy"), Y_pinn)
    
    metadata = {
        "grid_size": grid_size,
        "features": CYTOKINE_NAMES,
        "timesteps": N_TIMESTEPS,
        "window_size": window
    }
    with open(os.path.join(out_path, "metadata.json"), 'w') as f:
        json.dump(metadata, f, indent=4)
    
    print(f"Done. Saved all tensors to {out_path}\n")

In [8]:
if __name__ == "__main__":
    if not os.path.exists(BASE_LATTICE_DIR):
        print(f"Error: Directory not found at {BASE_LATTICE_DIR}")
    else:
        all_entries = os.listdir(BASE_LATTICE_DIR)
        resolution_folders = [d for d in all_entries if os.path.isdir(os.path.join(BASE_LATTICE_DIR, d))]
        
        for folder in sorted(resolution_folders):
            process_mesh_resolution(folder)

Processing LatticeData(100x100) (Grid: 100x100)
Done. Saved all tensors to C:\Users\jkowa\OneDrive\Pulpit\burns\preprocessed\100x100

Processing LatticeData(250x250) (Grid: 250x250)
Done. Saved all tensors to C:\Users\jkowa\OneDrive\Pulpit\burns\preprocessed\250x250

Processing LatticeData(500x500) (Grid: 500x500)
Done. Saved all tensors to C:\Users\jkowa\OneDrive\Pulpit\burns\preprocessed\500x500

Processing LatticeData(50x50) (Grid: 50x50)
Done. Saved all tensors to C:\Users\jkowa\OneDrive\Pulpit\burns\preprocessed\50x50

